# ADBE Annual Economic Value

The prior notebooks built a trustworthy foundation: raw SEC Company Facts become canonical fundamentals, then deterministic owner-economics and capital-efficiency metrics. This notebook adds the first *interpretation* layer on top of that foundation.

For each completed fiscal year it answers one question:

> Did Adobe's underlying per-share economics improve, stay roughly stable, or deteriorate during this year?

Everything here is deterministic. No new external data is fetched: the economic-value layer consumes the Feature 1 rows and classifies each year with explicit, documented rules. The classification is never an opaque score; every verdict carries named drivers you can read.

Two ideas do the heavy lifting:

1. **LEVEL vs CHANGE.** A level is a point-in-time characteristic (operating margin, FCF margin, ROIC, net cash). A change is a year-over-year movement (revenue growth, FCF/share growth, margin change, ROIC change). We never blend the two.
2. **Per-share economics outrank aggregate growth.** A bigger company is not the same as a richer owner. FCF *per share* is weighted above aggregate FCF, and buybacks or dilution are read explicitly.

> Set an SEC User-Agent identifying your application and an administrative contact. Replace the sample contact below before making live requests.

In [1]:
import os

from owner_lens import SecClient, economic_value_from_facts, format_economic_value_view

USER_AGENT = os.environ.get("OWNER_LENS_SEC_USER_AGENT", "OwnerLens admin@example.com")

raw = SecClient(USER_AGENT).retrieve_company_facts("ADBE").raw_facts
snapshots = economic_value_from_facts(raw)
years = [snap.fiscal_year for snap in snapshots]
years

[2025, 2024, 2023, 2022, 2021]

## The compact owner-oriented view

One row per fiscal year, newest first. `RevGr` is revenue growth, `OpMgΔ` is the operating-margin change in percentage points, `FCF/shGr` is free-cash-flow-per-share growth, `ShGr` is diluted-share-count growth (positive is dilution), and `Class` is the deterministic verdict.

The earliest displayed year shows `INSUFFICIENT_DATA`: its change signals need a prior year that is outside the display window. That is the correct answer, not a failure.

In [2]:
print(format_economic_value_view(snapshots))

  FY    RevGr    OpMgΔ    FCFGr  FCF/shGr     ShGr    ROIC    ROICΔ         NetCash  Class           
-----------------------------------------------------------------------------------------------------
2025   +10.5%    +5.3%   +25.1%    +31.8%    -5.0%   61.6%   +16.8%    +385,000,000  IMPROVING       
      drivers: FCF_PER_SHARE_STRONG_GROWTH, ROIC_EXPANDED, ROIC_SUSTAINED_HIGH, OPERATING_MARGIN_EXPANDED, FCF_MARGIN_EXPANDED, SHARE_COUNT_DECLINED, NET_CASH_DETERIORATED, REVENUE_MATERIAL_GROWTH, FCF_MATERIAL_GROWTH
2024   +10.8%    -2.9%   +13.4%    +15.8%    -2.0%   44.8%    +1.2%  +2,258,000,000  IMPROVING       
      drivers: FCF_PER_SHARE_STRONG_GROWTH, ROIC_SUSTAINED_HIGH, OPERATING_MARGIN_CONTRACTED, SHARE_COUNT_DECLINED, NET_CASH_DETERIORATED, REVENUE_MATERIAL_GROWTH, FCF_MATERIAL_GROWTH
2023   +10.2%    -0.4%    -6.1%     -3.7%    -2.5%   43.5%    +5.2%  +4,208,000,000  IMPROVING       
      drivers: ROIC_EXPANDED, ROIC_SUSTAINED_HIGH, FCF_MARGIN_CONTRACTED, SHARE_COUNT_DE

## Why each year got its classification

The drivers are the whole point of the transparency requirement. A year is not "improving" because a black box said so; it is improving because, for example, FCF per share grew strongly while the share count declined and ROIC stayed high. Reading the drivers teaches the economics.

In [3]:
for snap in snapshots:
    codes = ", ".join(driver.value for driver in snap.drivers) or "none"
    print(f"FY{snap.fiscal_year}  {snap.classification.value}")
    print(f"    {codes}")

FY2025  IMPROVING
    FCF_PER_SHARE_STRONG_GROWTH, ROIC_EXPANDED, ROIC_SUSTAINED_HIGH, OPERATING_MARGIN_EXPANDED, FCF_MARGIN_EXPANDED, SHARE_COUNT_DECLINED, NET_CASH_DETERIORATED, REVENUE_MATERIAL_GROWTH, FCF_MATERIAL_GROWTH
FY2024  IMPROVING
    FCF_PER_SHARE_STRONG_GROWTH, ROIC_SUSTAINED_HIGH, OPERATING_MARGIN_CONTRACTED, SHARE_COUNT_DECLINED, NET_CASH_DETERIORATED, REVENUE_MATERIAL_GROWTH, FCF_MATERIAL_GROWTH
FY2023  IMPROVING
    ROIC_EXPANDED, ROIC_SUSTAINED_HIGH, FCF_MARGIN_CONTRACTED, SHARE_COUNT_DECLINED, NET_CASH_IMPROVED, REVENUE_MATERIAL_GROWTH, FCF_DECLINE
FY2022  IMPROVING
    FCF_PER_SHARE_STRONG_GROWTH, ROIC_SUSTAINED_HIGH, OPERATING_MARGIN_CONTRACTED, FCF_MARGIN_CONTRACTED, SHARE_COUNT_DECLINED, NET_CASH_IMPROVED, REVENUE_MATERIAL_GROWTH, FCF_MATERIAL_GROWTH
FY2021  INSUFFICIENT_DATA
    INSUFFICIENT_PRIOR_YEAR_DATA


## Levels versus changes, side by side

The snapshot keeps levels and changes as distinct named fields. Here they are laid out together so the distinction is concrete: the left block is *where the business is*, the right block is *how it moved this year*.

In [4]:
def pct(value):
    return f"{value * 100:+.1f}%" if value is not None else "n/a"


def lvl(value):
    return f"{value * 100:.1f}%" if value is not None else "n/a"


for snap in snapshots:
    print(f"FY{snap.fiscal_year}")
    print(
        "    LEVEL   "
        f"op_margin={lvl(snap.operating_margin)}  "
        f"fcf_margin={lvl(snap.fcf_margin)}  "
        f"roic={lvl(snap.roic)}  "
        f"net_cash={snap.net_cash_or_debt}"
    )
    print(
        "    CHANGE  "
        f"rev_growth={pct(snap.revenue_growth)}  "
        f"fcf/sh_growth={pct(snap.fcf_per_share_growth)}  "
        f"share_growth={pct(snap.diluted_share_growth)}  "
        f"roic_change={pct(snap.roic_change)}"
    )

FY2025
    LEVEL   op_margin=36.6%  fcf_margin=41.4%  roic=61.6%  net_cash=385000000
    CHANGE  rev_growth=+10.5%  fcf/sh_growth=+31.8%  share_growth=-5.0%  roic_change=+16.8%
FY2024
    LEVEL   op_margin=31.3%  fcf_margin=36.6%  roic=44.8%  net_cash=2258000000
    CHANGE  rev_growth=+10.8%  fcf/sh_growth=+15.8%  share_growth=-2.0%  roic_change=+1.2%
FY2023
    LEVEL   op_margin=34.3%  fcf_margin=35.8%  roic=43.5%  net_cash=4208000000
    CHANGE  rev_growth=+10.2%  fcf/sh_growth=-3.7%  share_growth=-2.5%  roic_change=+5.2%
FY2022
    LEVEL   op_margin=34.6%  fcf_margin=42.0%  roic=38.3%  net_cash=1967000000
    CHANGE  rev_growth=+11.5%  fcf/sh_growth=+9.8%  share_growth=-2.1%  roic_change=-1.7%
FY2021
    LEVEL   op_margin=36.8%  fcf_margin=43.6%  roic=40.0%  net_cash=1675000000
    CHANGE  rev_growth=n/a  fcf/sh_growth=n/a  share_growth=n/a  roic_change=n/a


## What this layer is, and is not

It **is** a transparent judgment of whether the observable economics attributable to an owner improved during a single year, built only from trusted Feature 1 facts.

It is **not** a claim about intrinsic value, a valuation, a multi-year compounding view, or a 0-100 score. Those are deliberately out of scope. The thresholds that separate "material" from "noise" are centralized, named, and intentionally imprecise; they are meant to be economically intuitive and easy to revisit, not tuned to make Adobe look good.

This is the foundation for a later multi-year compounding view.